# Analysis: RAG vs LoRA for symptom-to-disease prediction

Reads the aggregated results written by `python -m medqa.evaluate --aggregate` and the per-run files,
and produces the tables and figures for the README. Set `OUTPUT_DIR` to the experiment's output folder.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Local "outputs" by default; on Colab use the Drive path automatically.
COLAB_OUTPUT = Path("/content/drive/MyDrive/Projects/medqa-lora-vs-rag/outputs")
OUTPUT_DIR = COLAB_OUTPUT if COLAB_OUTPUT.exists() else Path("outputs")
print("reading results from:", OUTPUT_DIR)
DEFAULT_K = 5
DEFAULT_RETRIEVER = "jaccard"

ARM_LABELS = {
    "base_zero_shot": "Base zero-shot",
    "base_rag": "Base + RAG",
    "lora": "LoRA",
    "lora_rag": "LoRA + RAG",
    "retrieval_only": "Retrieval-only",
}
# One fixed colour per arm, never reassigned when an arm is missing.
ARM_COLORS = {
    "base_zero_shot": "#2a78d6",
    "base_rag": "#eb6834",
    "lora": "#1baf7a",
    "lora_rag": "#eda100",
    "retrieval_only": "#e87ba4",
}
INK = "#52514e"

plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "#e6e5e1", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
})


def mean_std(frame: pd.DataFrame, metric: str, digits: int = 3) -> pd.Series:
    """Format 'mean ± std' per row, or just the mean when std is missing."""
    mean, std = frame[f"{metric}_mean"], frame[f"{metric}_std"]
    return [f"{m:.{digits}f} ± {s:.{digits}f}" if pd.notna(s) else f"{m:.{digits}f}" for m, s in zip(mean, std)]

## Load everything

In [ ]:
results = pd.read_csv(OUTPUT_DIR / "results.csv")
results = results[results["split"] == "test"]
mmlu = pd.read_csv(OUTPUT_DIR / "mmlu.csv") if (OUTPUT_DIR / "mmlu.csv").exists() else pd.DataFrame()
stats = json.loads((OUTPUT_DIR / "splits" / "stats.json").read_text())
train_stats = [json.loads(p.read_text()) for p in sorted(OUTPUT_DIR.glob("adapters/*/train_stats.json"))]

ceiling = stats["splits"]["test"]["accuracy_ceiling"]
print("models:", results["model"].unique().tolist())
print("test rows:", stats["splits"]["test"]["rows"], "| accuracy ceiling:", ceiling)
print("adapters with stats:", len(train_stats))
results

## Table 1: headline comparison at the default k

In [ ]:
def headline_rows(frame: pd.DataFrame) -> pd.DataFrame:
    """One row per arm: the default retriever and k for RAG arms, the only row otherwise."""
    rag = frame["retriever"].notna()
    picked = frame[~rag | ((frame["retriever"] == DEFAULT_RETRIEVER) & (frame["k"] == DEFAULT_K))]
    picked = picked.set_index("arm").reindex(list(ARM_LABELS)).dropna(subset=["exact_match_mean"]).reset_index()
    return picked


headline = headline_rows(results)
table1 = pd.DataFrame({
    "Arm": headline["arm"].map(ARM_LABELS),
    "Exact match": mean_std(headline, "exact_match"),
    "Semantic": mean_std(headline, "semantic"),
    "ROUGE-L": mean_std(headline, "rouge_l"),
    "Recall@k": [f"{v:.3f}" if pd.notna(v) else "n/a" for v in headline["recall_at_k_mean"]],
    "Latency (ms)": headline["latency_ms_mean"].round(1),
    "Prompt tokens": headline["prompt_tokens_mean"].round(0),
    "Runs": headline["n_runs"],
})
print(f"accuracy ceiling on the test split: {ceiling:.3f}")
table1

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.set_axisbelow(True)
y = range(len(headline))
ax.barh(y, headline["exact_match_mean"], xerr=headline["exact_match_std"].fillna(0),
        color=[ARM_COLORS[a] for a in headline["arm"]], height=0.55, error_kw={"ecolor": INK, "capsize": 3})
ax.set_yticks(list(y), headline["arm"].map(ARM_LABELS))
ax.invert_yaxis()
ax.axvline(ceiling, color=INK, linestyle="--", linewidth=1)
ax.text(ceiling, -0.6, f"ceiling {ceiling:.2f}", color=INK, ha="right", va="bottom", fontsize=9)
for yi, v in zip(y, headline["exact_match_mean"]):
    ax.text(v + 0.01, yi, f"{v:.3f}", va="center", color=INK, fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("Exact-match accuracy (test)")
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_title("Accuracy by arm, mean ± std over seeds", loc="left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_accuracy_by_arm.png", bbox_inches="tight")

## Table 2: effect of k

In [ ]:
with_k = results[results["retriever"] == DEFAULT_RETRIEVER]
table2 = with_k.pivot_table(index="arm", columns="k", values="exact_match_mean").reindex(
    [a for a in ARM_LABELS if a in with_k["arm"].values])
table2.index = table2.index.map(ARM_LABELS)
table2.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.6))
for arm in ("retrieval_only", "base_rag", "lora_rag"):
    rows = with_k[with_k["arm"] == arm].sort_values("k")
    if len(rows):
        ax.plot(rows["k"], rows["exact_match_mean"], marker="o", markersize=6, linewidth=2,
                color=ARM_COLORS[arm], label=ARM_LABELS[arm])
for arm in ("base_zero_shot", "lora"):
    row = results[results["arm"] == arm]
    if len(row):
        ax.axhline(row["exact_match_mean"].iloc[0], color=ARM_COLORS[arm], linewidth=1.5, linestyle=":",
                   label=f"{ARM_LABELS[arm]} (no retrieval)")
ax.axhline(ceiling, color=INK, linestyle="--", linewidth=1)
ax.set_xlabel("k retrieved cases"); ax.set_ylabel("Exact-match accuracy")
ax.set_xticks(sorted(with_k["k"].dropna().unique()))
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8, loc="lower right")
ax.set_title(f"Accuracy vs k ({DEFAULT_RETRIEVER} retriever)", loc="left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "fig_accuracy_vs_k.png", bbox_inches="tight")

## Table 3: general-knowledge retention (MMLU)

RAG leaves the weights unchanged, so the base row applies to Base + RAG and the LoRA row to LoRA + RAG.

In [ ]:
if len(mmlu):
    table3 = pd.DataFrame({
        "Weights": mmlu["weights"],
        "MMLU accuracy": mean_std(mmlu, "mmlu_accuracy"),
        "Runs": mmlu["n_runs"],
    })
    display(table3)
    fig, ax = plt.subplots(figsize=(4.5, 3))
    colors = [ARM_COLORS["base_zero_shot"] if w == "base" else ARM_COLORS["lora"] for w in mmlu["weights"]]
    ax.bar(mmlu["weights"], mmlu["mmlu_accuracy_mean"], yerr=mmlu["mmlu_accuracy_std"].fillna(0),
           color=colors, width=0.5, error_kw={"ecolor": INK, "capsize": 3})
    for x, v in zip(mmlu["weights"], mmlu["mmlu_accuracy_mean"]):
        ax.text(x, v + 0.01, f"{v:.3f}", ha="center", color=INK, fontsize=9)
    ax.axhline(0.25, color=INK, linestyle="--", linewidth=1)
    ax.text(0.5, 0.26, "chance", color=INK, fontsize=8, transform=ax.get_yaxis_transform())
    ax.set_ylim(0, max(0.6, mmlu["mmlu_accuracy_mean"].max() + 0.1))
    ax.set_ylabel("MMLU accuracy (250 questions)")
    ax.set_title("General-knowledge retention", loc="left")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_mmlu.png", bbox_inches="tight")
else:
    print("no mmlu.csv yet; run python -m medqa.evaluate --mmlu then --aggregate")

## Table 4: cost

In [ ]:
training = pd.DataFrame([{
    "seed": s["seed"], "steps": s["steps"], "train_minutes": s["wall_clock_s"] / 60,
    "peak_gpu_gb": s["peak_gpu_memory_gb"], "final_train_loss": s["train_loss"], "val_loss": s["val_loss"],
} for s in train_stats])
if len(training):
    display(training.round(3))
    print(f"training: {training['train_minutes'].mean():.1f} ± {training['train_minutes'].std():.1f} min per seed, "
          f"peak GPU memory {training['peak_gpu_gb'].max():.1f} GB")

table4 = pd.DataFrame({
    "Arm": headline["arm"].map(ARM_LABELS),
    "Latency per example (ms)": headline["latency_ms_mean"].round(1),
    "Prompt tokens": headline["prompt_tokens_mean"].round(0),
    "Training needed": ["yes" if a in ("lora", "lora_rag") else "no" for a in headline["arm"]],
})
table4

In [ ]:
if train_stats:
    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    seed_colors = ["#2a78d6", "#eb6834", "#1baf7a"]
    for color, s in zip(seed_colors, train_stats):
        history = pd.DataFrame([h for h in s["log_history"] if "loss" in h])
        ax.plot(history["step"], history["loss"], linewidth=1.5, color=color, label=f"seed {s['seed']}")
        evals = pd.DataFrame([h for h in s["log_history"] if "eval_loss" in h])
        if len(evals):
            ax.plot(evals["step"], evals["eval_loss"], marker="o", markersize=6, linestyle="none", color=color)
    ax.set_xlabel("step"); ax.set_ylabel("loss (disease tokens only)")
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("LoRA training loss; dots are validation loss per epoch", loc="left")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "fig_training_loss.png", bbox_inches="tight")

## Error analysis: where do LoRA and RAG disagree?

Per-example comparison between the LoRA arm (seed 42) and Base + RAG at the default k.

In [ ]:
def load_predictions(name: str) -> pd.DataFrame | None:
    path = OUTPUT_DIR / "predictions" / f"{name}.csv"
    return pd.read_csv(path) if path.exists() else None


lora_pred = load_predictions("lora_seed42")
rag_pred = load_predictions(f"base_rag_{DEFAULT_RETRIEVER}_k{DEFAULT_K}")
if lora_pred is not None and rag_pred is not None:
    # Both files list the test split in the same order; align by position, since
    # ambiguous symptom sets repeat the same symptom text with different diseases.
    assert len(lora_pred) == len(rag_pred)
    both = pd.DataFrame({
        "symptom_text": lora_pred["symptom_text"], "disease": lora_pred["disease"],
        "prediction_lora": lora_pred["prediction"], "exact_match_lora": lora_pred["exact_match"],
        "prediction_rag": rag_pred["prediction"], "exact_match_rag": rag_pred["exact_match"],
        "retrieved_labels": rag_pred["retrieved_labels"],
    })
    outcome = pd.crosstab(both["exact_match_lora"], both["exact_match_rag"],
                          rownames=["LoRA correct"], colnames=["Base + RAG correct"])
    display(outcome)
    print("\nLoRA right, RAG wrong:")
    display(both[both["exact_match_lora"] & ~both["exact_match_rag"]].head(5))
    print("\nRAG right, LoRA wrong:")
    display(both[~both["exact_match_lora"] & both["exact_match_rag"]].head(5))
else:
    print("need predictions/lora_seed42.csv and the base_rag file first")

## Ambiguity accounting

In [ ]:
test_stats = stats["splits"]["test"]
print(f"test rows: {test_stats['rows']}, symptom sets: {test_stats['symptom_sets']}, "
      f"ambiguous sets: {test_stats['ambiguous_symptom_sets']}")
print(f"best possible exact match: {ceiling:.3f}")
if lora_pred is not None:
    print(f"LoRA seed 42 reaches {lora_pred['exact_match'].mean():.3f}, "
          f"i.e. {lora_pred['exact_match'].mean() / ceiling:.1%} of the ceiling")